In [1]:
from input_pipeline import preprocess_dataset, get_masked_dataset, batch_sampler
import xarray as xr
import jax.numpy as jnp
import jax
import chex
import numpy as np
import flax
import rich
import flax.linen as nn
import optax

### Load, Normalization & Masking

In [2]:
ds = xr.load_dataarray('../data/SiC_100x10.nc')
filtered_ds = preprocess_dataset(ds, verbose = True)
masked_ds = get_masked_dataset(filtered_ds)

Dropped 20 spectra


In [19]:
# Transformer model, following "Formal algorithms for transformers" [arXiv:2207.09238 [cs.LG]] 

class LinearProjection(nn.Module):
    """Embedding layer for transformer model"""
    embedding_dim: int

    @nn.compact
    def __call__(self, x):
        x = nn.Dense(self.embedding_dim, use_bias=False)(x)
        x = nn.LayerNorm()(x)
        return x
    
class FFBlock(nn.Module):
    """Feed-forward block for transformer model"""
    embedding_dim: int
    dropout_rate: float = 0.1

    @nn.compact
    def __call__(self, x, training: bool):
        x = nn.Dense(4*self.embedding_dim)(x)
        x = nn.relu(x)
        x = nn.Dense(self.embedding_dim)(x)
        x = nn.Dropout(self.dropout_rate, deterministic = not training)(x)
        return x

class TransformerEncoderLayer(nn.Module):
    """Transformer encoder layer"""
    embedding_dim: int
    num_heads: int
    dropout_rate: float = 0.1

    @nn.compact
    def __call__(self, x, training: bool):
        # Multi-head attention
        x = nn.LayerNorm()(x)
        x = nn.MultiHeadDotProductAttention(
            num_heads = self.num_heads,
            qkv_features = self.embedding_dim,
            dropout_rate = self.dropout_rate
        )(x, deterministic = not training)
        x = x + nn.Dense(self.embedding_dim)(x)
        x = nn.Dropout(self.dropout_rate, deterministic = not training)(x)
        x = nn.LayerNorm()(x)
        x = FFBlock(self.embedding_dim, self.dropout_rate)(x, training = training)
        x = x + nn.Dense(self.embedding_dim)(x)
        return x
    
class SpectraFormer(nn.Module):
    embedding_dim: int
    num_heads: int
    num_layers: int
    dropout_rate: float = 0.1

    @nn.compact
    def __call__(self, counts, wave_number, training: bool):
        emb_counts = LinearProjection(self.embedding_dim)(counts)
        emb_wave_number = LinearProjection(self.embedding_dim)(wave_number)
        x = emb_counts + emb_wave_number
        for _ in range(self.num_layers):
            x = TransformerEncoderLayer(self.embedding_dim, self.num_heads, self.dropout_rate)(x, training = training)
        x = nn.LayerNorm()(x)
        x = nn.Dense(1)(x)
        return x

In [27]:
batch_size = 32
train_iter = batch_sampler(filtered_ds, masked_ds, batch_size = batch_size, shuffle = True, drop_last = True)
batch = next(train_iter)

model = SpectraFormer(embedding_dim = 64, num_heads = 8, num_layers = 4)
root_key = jax.random.key(seed=0)
main_key, params_key, dropout_key = jax.random.split(key=root_key, num=3)

variables = model.init(params_key, counts = batch['masked_spectra'][0], wave_number = batch['wave_number'][0], training = False)
params = variables['params']

In [28]:
# Print model summary
# print(model.tabulate(init_rng, counts = batch['masked_spectra'][0], wave_number = batch['wave_number'][0], training = False, compute_flops=True, compute_vjp_flops=True))

In [29]:
from flax.training import train_state

class TrainState(train_state.TrainState):
  key: jax.Array

state = TrainState.create(
  apply_fn=model.apply,
  params=params,
  key=dropout_key,
  tx=optax.adam(1e-3)
)

In [30]:
state.apply_fn({'params': params}, counts = batch['masked_spectra'], wave_number = batch['wave_number'], training = True, rngs = {'dropout': dropout_key}).shape

(32, 1015, 1)

In [33]:
@jax.jit
def train_step(state: TrainState, batch, dropout_key):
  dropout_train_key = jax.random.fold_in(key=dropout_key, data=state.step)
  def loss_fn(params):
    pred_spectra = state.apply_fn(
      {'params': params},
      counts = batch['masked_spectra'], wave_number = batch['wave_number'],
      training=True,
      rngs={'dropout': dropout_train_key}
      )
    loss = optax.squared_error(pred_spectra, batch['spectra']).mean()
    return loss
  grad_fn = jax.value_and_grad(loss_fn)
  loss, grads = grad_fn(state.params)
  state = state.apply_gradients(grads=grads)
  return state

In [36]:
from time import perf_counter
train_iter = batch_sampler(filtered_ds, masked_ds, batch_size = batch_size, shuffle = True, drop_last = True)
times = []
for batch in train_iter:
    _dt = perf_counter()
    state = train_step(state, batch, dropout_key)
    _dt = perf_counter() - _dt
    times.append(_dt)
times = np.array(times)
print(f"Batch size: {batch_size}, throughput: {batch_size / times.mean() * 1e-3:.2f} ksamples/s")

Batch size: 32, throughput: 0.21 ksamples/s


In [10]:


# Measure throughput at different batch sizes
batch_sizes = [8, 16, 32, 64, 128]

for batch_size in batch_sizes:
    train_iter = batch_sampler(filtered_ds, masked_ds, batch_size = batch_size, shuffle = True, drop_last = True)
    rng = jax.random.PRNGKey(0)
    rng, init_rng = jax.random.split(rng)
    params = model.init(init_rng, counts = batch['masked_spectra'][0], wave_number = batch['wave_number'][0], training = False)
    # Warmup
    batch = next(train_iter)
    apply_fn = jax.jit(model.apply, static_argnums = 2)
    apply_fn(params, batch['masked_spectra'], batch['wave_number'], True, rngs = {'dropout': rng})
    
    # Measure throughput
    times = []
    for batch in train_iter:
        _dt = perf_counter()
        out = apply_fn(params, batch['masked_spectra'], batch['wave_number'], True, rngs = {'dropout': rng})
        _dt = perf_counter() - _dt
        times.append(_dt)
    times = np.array(times)
    print(f"Batch size: {batch_size}, throughput: {batch_size / times.mean() * 1e-3:.2f} ksamples/s")

ValueError: Non-hashable static arguments are not supported. An error occurred during a call to 'apply' while trying to hash an object of type <class 'jaxlib.xla_extension.ArrayImpl'>, [[1278.2266]
 [1279.9258]
 [1281.627 ]
 ...
 [2817.6406]
 [2819.002 ]
 [2820.3623]]. The error was:
TypeError: unhashable type: 'ArrayImpl'


In [15]:
type(batch['masked_spectra'])

jaxlib.xla_extension.ArrayImpl